<!-- cabecera-entorno -->
## Antes de empezar

**Clase 5 · EDA: bivariado y multivariado** — Bloque 2 · Demo. Este cuaderno se recorre **por su
cuenta**: explica cada concepto antes de usarlo, y el profesor circula por el salón resolviendo
dudas. No hay que esperar a que alguien lo dicte.

**La rutina de siempre:** `git pull` antes de clase, y el entorno virtual activo (`(.venv)` en la
terminal). Si va a modificar este archivo, trabaje sobre una copia: duplique `demo.ipynb` como
`demo_mio.ipynb` y edite el duplicado. Así `git pull` nunca le reclama.

**Si la celda de abajo falla, no siga:** la respuesta está en el manual del entorno,
[`../INSTALACION.md`](../INSTALACION.md).

| Si ve esto | Qué pasó | Dónde se arregla |
|------------|----------|------------------|
| `ModuleNotFoundError` | El entorno virtual no está activo, o VSCode eligió otro intérprete | Manual, secciones 6.3 y 8.4, y problema 5 |
| `FileNotFoundError` al leer el CSV | El notebook se abrió desde otra carpeta, o falta hacer `git pull` | Manual, problema 6 |
| El kernel no aparece en VSCode | Falta la extensión Jupyter o `ipykernel` dentro del entorno | Manual, problema 4 |

In [ ]:
# Verificación del entorno. Si algo falla aquí, la solución está en ../INSTALACION.md
import sys
from pathlib import Path

try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Falta la librería '{error.name}'. Active el entorno virtual y seleccione el intérprete "
        ".venv en VSCode (Ctrl+Shift+P > Python: Select Interpreter), luego reinicie el kernel. "
        "Ver ../INSTALACION.md, problema 5."
    ) from error

print("Intérprete:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este no parece el Python del entorno virtual. En VSCode: Ctrl+Shift+P >",
          "'Python: Select Interpreter' > el que dice .venv, y reinicie el kernel.")

RUTA_VERIFICACION = "../datos/HISTORICO_CONSUMO.csv"
if Path(RUTA_VERIFICACION).exists():
    print("Datos: encontrados en", RUTA_VERIFICACION)
else:
    print("FALTA el archivo", RUTA_VERIFICACION, "- abra en VSCode la carpeta raíz del curso",
          "y ejecute 'git pull'. Ver ../INSTALACION.md, problema 6.")

# Clase 5 · Demo — EDA: análisis bivariado y multivariado

> **Dónde vamos.** Cuarta y última de las clases de **EDA**, análisis exploratorio de datos (2, 3,
> 4 y 5). El verbo de hoy es **relacionar**: unas variables contra otras, para dejar listo el
> terreno donde se decide. Al terminar hoy habrán hecho un **EDA completo**, que es lo que se entrega
> en la clase 6. El marco completo está en el cuaderno de la clase 2, sección *Qué estamos
> haciendo: EDA*.

**Dataset:** `../datos/HISTORICO_CONSUMO.csv`, el consumo de agua de Empocaldas. El mismo de la
clase 4, a propósito: la limpieza ya la conocen, así que todo el tiempo de hoy se va en lo nuevo.

## Cómo se usa este cuaderno

Está escrito para que usted avance solo. Cada bloque de código viene precedido de la explicación del
concepto que usa, y cada término nuevo se define la primera vez que aparece. **Hay que leer antes de
ejecutar.**

**El recorrido:**

| Sección | De qué va | Qué se lleva |
|---------|-----------|--------------|
| 0 y 1 | El dataset, las librerías, la carga y su verificación | El dato en memoria y la certeza de que está bien |
| 2 | Cómo se lee un scatter: tendencia, ruido, forma, grupos | El paso 2 del marco |
| 3 | Qué mide el coeficiente r, por dentro, y qué **no** mide | El paso 3 del marco |
| 3.4 a 3.7 | Las cuatro cosas que r **no** ve, provocadas con datos | Saber cuándo el número miente |
| 4 | Matriz de correlación, qué hace `.corr()` por dentro, heatmap | Todos los pares de un vistazo |
| 5 | Categórica contra numérica: boxplot y barras | El otro caso de la tabla de gráficos |
| 6 | Pair plot: el salto a multivariado | Todos los pares, dibujados |
| 7 | La paradoja de Simpson, fabricada y luego real | El paso 5 del marco |
| 8 | Interrogar: correlación contra causalidad | El paso 4 del marco |

**Aquí no hay nada que teclear.** Todo el código está escrito y ejecutable: usted lo corre, mira la
salida y lee la explicación que está justo encima. Escribir código es el bloque 3, con el reto, y es
lo que se entrega.

**Las catorce preguntas de interpretación** son lo que sí le toca a usted. No llevan código: se
responden escribiendo en español en la celda, debajo de *Tu respuesta:*. Cada una trae un bloque
plegable *"Comparar con la respuesta esperada"*. **Escriba la suya primero y ábralo después.**
Abrirlo antes no le ahorra nada: lo que se evalúa en el reto y en la sustentación es que usted sepa
mirar un resultado y decir qué significa, no que sepa reproducir un cálculo.

**Las secciones "Para entender qué está pasando"** explican el fundamento: qué mide realmente una
correlación, qué hace la herramienta por dentro, por qué un gráfico puede engañar. Son las que
convierten esto en análisis en vez de recetas. Si ya las tiene claras, se pueden saltar sin perder el
hilo del código; si no, son la parte que más va a servirle en los momentos evaluativos.

---

### Lo que este cuaderno da por sabido

No se repite aquí nada de lo que ya se explicó. Si algo de esta lista no le suena, vuelva al cuaderno
que lo enseña **antes** de seguir: hoy todo se apoya en eso.

| Lo que se da por sabido | Dónde se explicó |
|-------------------------|------------------|
| Qué es una librería, un alias, un DataFrame, una Series, el índice, un dtype | Clase 2, demo, secciones 1 a 5 |
| `read_csv`, `head`, `info`, `describe`, `shape` | Clase 2, demo, secciones 2 y 3 |
| Máscara booleana, `&`, `\|`, `~`, `.isin()` | Clase 2, demo, secciones 7 a 11 |
| Nulos, `NaN`, conversión de tipos, valores inválidos de dominio | Clase 3, demo, pasos 1 a 6 |
| Media, mediana, desviación estándar, outliers y la regla 1.5xIQR | Clase 4, demo, secciones 3 y 6 |
| `groupby`, histograma y boxplot de una variable | Clase 4, demo, secciones 4 y 5 |

**Lo nuevo de hoy** es todo lo que involucra **dos** variables a la vez: la correlación y su
interpretación, la matriz y el heatmap, la comparación entre categorías, el pair plot y el chequeo de
Simpson. Eso sí va explicado desde cero.

---

### El marco bivariado de 5 pasos

Es el esqueleto de toda la clase, y el gemelo del marco univariado de la clase 4.

| Paso | Nombre | Qué se hace |
|------|--------|-------------|
| 1 | Tipar | ¿Qué tipo es cada una de las dos variables? |
| 2 | Graficar | El gráfico que dice la tabla. **Antes** de calcular nada |
| 3 | Cuantificar | r si son dos numéricas; diferencia de medias si hay una categórica |
| 4 | Interrogar | ¿Tiene sentido? ¿Hay confusora? ¿Podría ser espuria? |
| 5 | Segmentar | Chequeo de Simpson: ¿la relación sobrevive dentro de cada grupo? |

**El paso 1 decide el gráfico**, con esta tabla:

| Variable 1 | Variable 2 | Gráfico | Métrica |
|------------|------------|---------|---------|
| Numérica | Numérica | Diagrama de dispersión (scatter) | Coeficiente r |
| Categórica | Numérica | Boxplot, o barras de medias | Comparación de medias/medianas |
| Categórica | Categórica | Tabla cruzada, mapa de calor | Conteos y proporciones |

**El paso 2 va antes que el paso 3 a propósito.** Es la contramedida contra las tres trampas de la
sección 3.4: no linealidad, outliers y grupos escondidos.

---

## 0. El dataset, antes de tocarlo

Regla de la casa: nunca se ejecuta una línea de código sobre un dataset que no se sabe qué es.

| Campo | Valor |
|-------|-------|
| Qué mide | Consumo de agua y alcantarillado facturado por Empocaldas |
| Tamaño | 21.816 filas x 12 columnas antes de limpiar |
| Granularidad | Una fila = un municipio, un estrato, un mes, un año |
| Cobertura | 24 municipios de Caldas, 9 categorías de estrato, 2015-2023 |

**Qué es la granularidad.** El nivel de detalle de una fila: qué combinación de cosas hace que una
fila sea distinta de otra. Aquí una fila **no** es un cliente ni una factura: es un grupo de clientes
(los de un estrato, en un municipio, en un mes). Saberlo cambia todo lo que se puede decir después.

**La distinción que decidió la clase pasada y vuelve a decidir la de hoy:**

- `CONSUMO_ACUEDUCTO` es una **suma** del grupo. Crece si el grupo es grande, aunque cada suscriptor
  gaste poco.
- `PROMEDIO_ACUEDUCTO` es un **cociente** por suscriptor. No depende del tamaño del grupo.

Son dos preguntas distintas, y hoy vamos a medir con un número qué tan distintas son.

**Qué tiene de sucio.** Lo mismo de la clase 4: `AÑO` viene como `"2,015"`, los consumos y los
suscriptores traen punto como separador de miles, `PROMEDIO` trae coma, `NIT` y `RAZON SOCIAL` son
constantes, y hay 394 filas con faltantes. La celda de carga viene escrita.

---

## 1. Las librerías de hoy

**matplotlib** es la librería de dibujo de Python. Todo gráfico que vea en este curso lo dibuja
matplotlib por debajo, incluso cuando el que lo pide es otro. El alias convencional es `plt`, y viene
de `matplotlib.pyplot`, que es el submódulo con las funciones de dibujo.

**seaborn** (alias `sns`) se monta encima de matplotlib y agrega los gráficos **estadísticos**: el
mapa de calor, el boxplot por categoría, el scatter con línea de tendencia, el pair plot. Lo que en
matplotlib son veinte líneas, en seaborn es una. No reemplaza a matplotlib: lo usa. Por eso los
títulos y las etiquetas se siguen poniendo con `plt`.

**numpy** (alias `np`) es la aritmética por debajo de pandas: los arreglos de números. Hoy lo usamos
poco y de forma directa, para fabricar los ejemplos de la sección 3.4.

Las dos primeras líneas de configuración le dicen a pandas cómo imprimir las tablas, y las dos
últimas fijan el estilo de los gráficos para que todos veamos lo mismo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print('pandas', pd.__version__, '| seaborn', sns.__version__)

### 1.2 La carga, explicada

La celda de abajo es la misma de la clase 4 y hace cuatro cosas. Léala antes de ejecutarla:

1. **`dtype=str`** obliga a pandas a leerlo todo como texto. Suena absurdo, y es deliberado: si
   pandas adivina, un `"2,015"` puede terminar convertido en cualquier cosa. Primero se lee tal cual,
   después se convierte a mano.
2. **`drop` y `rename`** botan las dos columnas constantes y les ponen a las demás nombres que se
   puedan teclear sin tildes ni espacios.
3. **`dropna()`** elimina las filas con faltantes.
4. **La conversión de tipos**, columna por columna: se quita el separador de miles con
   `.str.replace(...)` y después se convierte con `.astype(...)`.

Fíjese en el detalle del separador: el año y los promedios usan **coma**, mientras que los consumos y
los suscriptores usan **punto**. En el mismo archivo. Es normal y es una de las razones por las que la
verificación de la celda siguiente no es opcional.

In [ ]:
# Este cuaderno vive en clase05/demo/, y el CSV dos carpetas más arriba, en datasets/
df = pd.read_csv('../datos/HISTORICO_CONSUMO.csv', dtype=str)

df = df.drop(columns=['NIT', 'RAZON SOCIAL'])
df = df.rename(columns={
    'AÑO': 'ANIO',
    'No. SUSCRIPTORES ACUEDUCTO': 'SUSCRIPTORES_ACUEDUCTO',
    'CONSUMO M3 ACUEDUCTO': 'CONSUMO_ACUEDUCTO',
    'PROMEDIO CONSUMO ACUEDUCTO': 'PROMEDIO_ACUEDUCTO',
    'No. SUSCRIPTORES ALCANTARILLADO': 'SUSCRIPTORES_ALCANTARILLADO',
    'CONSUMO M3 ALCANTARILLADO': 'CONSUMO_ALCANTARILLADO',
    'PROMEDIO CONSUMO ALCANTARILLADO': 'PROMEDIO_ALCANTARILLADO'
})
df = df.dropna()

df['ANIO'] = df['ANIO'].str.replace(',', '', regex=False).astype(int)
for col in ['SUSCRIPTORES_ACUEDUCTO', 'CONSUMO_ACUEDUCTO',
            'SUSCRIPTORES_ALCANTARILLADO', 'CONSUMO_ALCANTARILLADO']:
    df[col] = df[col].str.replace('.', '', regex=False).astype(int)
for col in ['PROMEDIO_ACUEDUCTO', 'PROMEDIO_ALCANTARILLADO']:
    df[col] = df[col].str.replace(',', '', regex=False).astype(float)

print(f'Filas y columnas: {df.shape}')
df.head()

### 1.3 La verificación, que es el punto de toda esta sección

**El concepto: hay errores que no lanzan ninguna excepción.** Si la conversión de miles fallara —si
un `164.863` se quedara en `164`— el cuaderno correría entero, los gráficos saldrían bonitos y todos
los resultados estarían mil veces mal. Nada en rojo, nada en la consola, y un análisis inservible.

Contra eso solo hay una defensa: **verificar contra un número que usted conozca de antemano.** El
máximo real de `CONSUMO_ACUEDUCTO` es 164.863 m3.

`assert condicion, 'mensaje'` es la forma corta de escribir esa verificación: si la condición es
falsa, detiene el cuaderno con el mensaje. Es la manera de que un error silencioso deje de serlo.

In [ ]:
assert df['CONSUMO_ACUEDUCTO'].max() > 100000, 'La conversión de miles falló'

print('Verificación superada. Máximo:', f"{df['CONSUMO_ACUEDUCTO'].max():,}", 'm3')
print('Columnas numéricas:', df.select_dtypes('number').columns.tolist())

---

## 2. Paso 2 del marco: graficar, antes de calcular

Primera pareja: `SUSCRIPTORES_ACUEDUCTO` contra `CONSUMO_ACUEDUCTO`. Paso 1, tipar: las dos son
numéricas, así que la tabla dice **diagrama de dispersión** (*scatter*).

**Qué es un diagrama de dispersión.** Un punto por fila, con una variable en el eje horizontal y la
otra en el vertical. No resume nada: muestra las 21.422 filas tal como son. Es la forma más honesta de
mirar una relación entre dos numéricas, y por eso va primero.

**Qué es `alpha`.** La transparencia de cada punto, de 0 (invisible) a 1 (opaco). Con 21.422 puntos
encimados, sin `alpha` se ve una mancha negra: no se distingue una zona con diez puntos de una con
diez mil. Con `alpha=0.15`, donde hay muchos encimados se ve oscuro y donde hay pocos se ve claro, y
el gráfico pasa a mostrar **densidad**, no solo posición.

La celda de abajo dibuja las dos versiones lado a lado. `plt.subplots(1, 2)` crea una figura con dos
ejes: la figura es el lienzo, y cada **eje** es un gráfico dentro de ese lienzo.

---

### Para entender qué está pasando · Cómo se lee un scatter

Un scatter no se mira: se interroga. Son cinco preguntas, siempre las mismas, y en este orden.

1. **¿Hay tendencia?** ¿La nube sube, baja, o no va a ninguna parte? La tendencia es el patrón que se
   mantiene a lo largo de todo el eje horizontal. Si al taparle la mitad derecha al gráfico usted
   podría adivinar más o menos dónde siguen los puntos, hay tendencia.
2. **¿Qué forma tiene?** Recta, curva, escalón, abanico. Esto es lo que decide si el coeficiente r que
   va a calcular en la sección 3 significa algo o no significa nada.
3. **¿Cuánto ruido hay alrededor de la tendencia?** El **ruido** es la variación que la tendencia no
   explica: qué tan lejos de la línea imaginaria caen los puntos. Poco ruido, puntos pegados a la
   línea, relación fuerte. Mucho ruido, nube gorda, relación débil aunque la tendencia exista. Ojo con
   la confusión más común: **una relación puede ser fuerte y tener pendiente casi plana, o ser débil y
   tener pendiente empinada.** La fuerza es qué tan pegados están los puntos, no qué tan inclinada
   está la línea.
4. **¿Hay puntos sueltos muy lejos del resto?** Son los outliers de la clase 4, ahora en dos
   dimensiones: un punto puede ser normal en cada variable por separado y absurdo en la combinación.
5. **¿Hay grupos?** Manchas separadas dentro de la misma nube. Cuando aparecen, casi siempre hay una
   variable categórica escondida detrás, y esa variable es la que manda. Es la sección 7.

**Y una advertencia de honestidad sobre lo que un scatter no puede decir:** ni la tendencia más limpia
del mundo demuestra que una variable **cause** la otra. Un scatter describe cómo se mueven juntas dos
columnas de una tabla. Nada más.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))

ejes[0].scatter(df['SUSCRIPTORES_ACUEDUCTO'], df['CONSUMO_ACUEDUCTO'], s=8)
ejes[0].set_xlabel('Suscriptores de acueducto')
ejes[0].set_ylabel('Consumo de acueducto (m3)')
ejes[0].set_title('Sin alpha: una mancha')

ejes[1].scatter(df['SUSCRIPTORES_ACUEDUCTO'], df['CONSUMO_ACUEDUCTO'], s=8, alpha=0.15)
ejes[1].set_xlabel('Suscriptores de acueducto')
ejes[1].set_ylabel('Consumo de acueducto (m3)')
ejes[1].set_title('Con alpha=0.15: se ve la densidad')

plt.tight_layout()
plt.show()

### La misma nube, con línea de tendencia

`sns.regplot()` hace el scatter y le encima la recta que mejor se ajusta a los puntos. La franja
alrededor de la recta es el **intervalo de confianza** de esa recta: qué tan segura está la estimación
de dónde va la línea. Ancha significa poca certeza; angosta, mucha.

Se dibuja sobre una **muestra** de 2.000 filas. `.sample(n, random_state=42)` toma n filas al azar;
`random_state` fija el azar para que a todos les salga exactamente la misma muestra. Sin eso, cada
ejecución daría un gráfico distinto y no habría nada que comparar entre compañeros.

In [ ]:
muestra = df.sample(2000, random_state=42)

plt.figure(figsize=(10, 6))
sns.regplot(x='SUSCRIPTORES_ACUEDUCTO', y='CONSUMO_ACUEDUCTO', data=muestra,
            scatter_kws={'alpha': 0.3, 's': 15},
            line_kws={'color': 'red'})
plt.xlabel('Suscriptores de acueducto')
plt.ylabel('Consumo de acueducto (m3)')
plt.title('Suscriptores vs consumo, con línea de tendencia (muestra de 2.000)')
plt.tight_layout()
plt.show()

**Pregunta de interpretación 1.** Antes de calcular ningún número: mirando la nube, ¿la relación es
positiva, negativa o nula? ¿Diría que es fuerte, moderada o débil? ¿Y por qué la nube se abre a medida
que avanza hacia la derecha?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Positiva y fuerte: la nube sube de izquierda a derecha y los puntos se agrupan alrededor de la recta.
Más suscriptores, más consumo total. Nada sorprendente: es casi contable.

Lo interesante es la forma de abanico. A la izquierda, con pocos suscriptores, los puntos están
apretados contra la recta; a la derecha se abren muchísimo. Eso significa que la relación es más
predecible en los grupos pequeños que en los grandes: cuando hay 20.000 suscriptores, el consumo total
puede ser muy distinto según de qué tipo de suscriptores se trate. Ese abanico es la primera pista de
que hay una tercera variable metida en la relación, y en la sección 7 le vamos a poner nombre.

</details>

---

## 3. Paso 3 del marco: cuantificar

Ahora sí el número. **El coeficiente de correlación de Pearson, r**, resume en un solo número qué tan
bien se ajusta una recta a la nube.

`serie_a.corr(serie_b)` lo calcula entre dos columnas.

**Cómo se lee:**

- Va de **-1 a +1**, siempre. Si a alguien le da 1,4, se equivocó en algo.
- El **signo** dice la dirección: positivo, las dos suben juntas; negativo, una sube y la otra baja.
- El **valor absoluto** dice la fuerza.

| \|r\| | Lectura |
|-------|---------|
| 0,0 - 0,3 | Débil |
| 0,3 - 0,7 | Moderada |
| 0,7 - 1,0 | Fuerte |

**Y lo que r NO es**, que es más importante:

1. **No es un porcentaje.** r = 0,5 no es "la mitad de la relación". No tiene unidades y no se
   interpreta como fracción.
2. **No mide cualquier relación: mide la relación lineal.** Una parábola perfecta da r cercano a 0.
3. **No es robusto.** Un solo outlier extremo lo mueve de forma dramática.
4. **No dice nada sobre causas.** Nada. Este punto tiene su propia sección.
5. **Con pocos datos no significa gran cosa.** Con n menor a 10, prácticamente nada.

Los puntos 2 y 3 se ven con sus ojos en la sección 3.4.

---

### Para entender qué está pasando · Qué mide r por dentro

Vale la pena, una sola vez en el semestre, ver de dónde sale el número. Después no hay que volver a
pensarlo, pero saberlo cambia cómo se interpreta.

**El problema que resuelve r.** Queremos decir si dos variables "se mueven juntas". La dificultad es
que están en unidades distintas: suscriptores son personas y consumo son metros cúbicos, y no se
pueden comparar directamente. Así que primero hay que ponerlas en la misma escala.

**Paso 1: estandarizar.** A cada valor se le resta la media de su columna y se divide por su
desviación estándar. El resultado se llama **puntaje z** (*z-score*), y dice *a cuántas desviaciones
estándar del promedio está este valor*. La media y la desviación estándar son de la clase 4; el
puntaje z es el uso más importante que van a tener.

Después de estandarizar, las dos columnas hablan el mismo idioma: un valor por encima de su promedio
es positivo, uno por debajo es negativo, y da igual si eran personas o metros cúbicos.

**Paso 2: multiplicar y promediar.** Para cada fila se multiplican los dos puntajes z:

- Fila **por encima del promedio en las dos** variables: negativo por negativo, no; positivo por
  positivo, **producto positivo**.
- Fila **por debajo del promedio en las dos**: negativo por negativo, **producto positivo** también.
- Fila **por encima en una y por debajo en la otra**: **producto negativo**.

r es el promedio de todos esos productos. Ahí está toda la intuición:

- Si las filas tienden a estar del mismo lado del promedio en las dos variables, los productos son
  mayoritariamente positivos y **r sale positivo**.
- Si tienden a estar en lados opuestos, los productos son negativos y **r sale negativo**.
- Si no hay ningún patrón, los positivos y los negativos se cancelan y **r sale cerca de cero**.

Y ahí también está la limitación: si la relación es una U, la mitad izquierda aporta productos
negativos y la derecha productos positivos, **se cancelan**, y r sale cero aunque la relación sea
perfecta. No es un defecto del cálculo: es que el cálculo mide otra cosa.

La celda de abajo lo hace a mano y compara con `.corr()`. Los dos números tienen que ser el mismo.

In [ ]:
col_a = df['SUSCRIPTORES_ACUEDUCTO'].astype(float)
col_b = df['CONSUMO_ACUEDUCTO'].astype(float)

# Paso 1: estandarizar. Cada valor, en desviaciones estándar respecto a su propio promedio.
za = (col_a - col_a.mean()) / col_a.std()
zb = (col_b - col_b.mean()) / col_b.std()

# Paso 2: multiplicar fila por fila y promediar
r_a_mano = (za * zb).sum() / (len(col_a) - 1)

print(f'r calculado a mano : {r_a_mano:.6f}')
print(f'r con .corr()      : {col_a.corr(col_b):.6f}')
print()
concordantes = ((za > 0) & (zb > 0)).sum() + ((za < 0) & (zb < 0)).sum()
discordantes = ((za > 0) & (zb < 0)).sum() + ((za < 0) & (zb > 0)).sum()

print(f'Filas del MISMO lado del promedio en las dos (producto positivo): {concordantes:,}')
print(f'Filas de lados OPUESTOS (producto negativo):                     {discordantes:,}')
print()
print('Como las primeras son bastante más que las segundas, r sale positivo.')

In [ ]:
r = df['SUSCRIPTORES_ACUEDUCTO'].corr(df['CONSUMO_ACUEDUCTO'])
print(f'r(suscriptores, consumo total) = {r:.3f}')

if abs(r) > 0.7:
    fuerza = 'FUERTE'
elif abs(r) > 0.3:
    fuerza = 'MODERADA'
else:
    fuerza = 'DÉBIL'

direccion = 'positiva' if r > 0 else 'negativa'
print(f'Lectura: correlación {fuerza} {direccion}.')

### 3.2 El par que importa de verdad

Ese 0,75 no sorprende a nadie: más suscriptores, más consumo total. Es casi una tautología.

La pregunta interesante es otra: **¿los grupos con más suscriptores gastan más agua POR suscriptor?**
Es decir, `SUSCRIPTORES_ACUEDUCTO` contra `PROMEDIO_ACUEDUCTO`.

In [ ]:
r_promedio = df['SUSCRIPTORES_ACUEDUCTO'].corr(df['PROMEDIO_ACUEDUCTO'])

print(f'r(suscriptores, consumo POR SUSCRIPTOR) = {r_promedio:.3f}')

Prácticamente cero.

Que un grupo tenga muchos suscriptores **no dice absolutamente nada** sobre cuánto gasta cada
suscriptor. Son dos preguntas distintas, y este dataset las separa en dos columnas.

Es la lección de la clase 4 —suma contra cociente— ahora medida con un número. Y es la razón por la
que el paso 1 del marco es "tipar": si uno no sabe si su variable es una suma o un cociente, el r que
calcule le va a responder otra pregunta.

Así se ve un r nulo al lado de uno de 0,75:

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))

ejes[0].scatter(muestra['SUSCRIPTORES_ACUEDUCTO'], muestra['CONSUMO_ACUEDUCTO'],
                s=12, alpha=0.4, color='steelblue')
ejes[0].set_xlabel('Suscriptores')
ejes[0].set_ylabel('Consumo TOTAL (m3)')
ejes[0].set_title(f'Correlación fuerte: r = {r:.3f}')

ejes[1].scatter(muestra['SUSCRIPTORES_ACUEDUCTO'], muestra['PROMEDIO_ACUEDUCTO'],
                s=12, alpha=0.4, color='coral')
ejes[1].set_xlabel('Suscriptores')
ejes[1].set_ylabel('Consumo POR SUSCRIPTOR (m3)')
ejes[1].set_title(f'Correlación nula: r = {r_promedio:.3f}')

plt.tight_layout()
plt.show()

### 3.4 Lo que r no ve (1): la curva

Aquí empieza el tramo de los errores provocados a propósito. Todos los de esta sección tienen la misma
firma: **el código corre, no hay nada en rojo, y el número está mal interpretado.**

La celda de abajo fabrica una relación **perfecta**: `y` es exactamente `x` al cuadrado. No hay ruido,
no hay azar, no falta un dato. Conociendo `x` se sabe `y` con certeza absoluta.

Mire qué dice r de esa relación perfecta.

In [ ]:
x_curva = np.linspace(-10, 10, 201)
y_curva = x_curva ** 2

r_curva = np.corrcoef(x_curva, y_curva)[0, 1]

plt.figure(figsize=(7, 5))
plt.scatter(x_curva, y_curva, s=14, color='seagreen')
plt.xlabel('x')
plt.ylabel('y = x al cuadrado')
plt.title(f'Relación perfecta... y sin embargo r = {r_curva:.3f}')
plt.tight_layout()
plt.show()

print(f'r = {r_curva:.3f}')
print('Relación perfecta, determinista, sin una pizca de azar. Y r dice que no hay nada.')

**Pregunta de interpretación 2.** ¿Por qué r vale cero si la relación es perfecta? Y si mañana su
scatter muestra una curva en vez de una recta, ¿qué hace?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Porque r de Pearson mide **relación lineal** y solo eso: qué tan bien una **recta** describe la nube.
En una parábola simétrica, la mitad izquierda baja y la mitad derecha sube; la mejor recta posible es
horizontal, y r sale cero. El número no está mal calculado: está respondiendo la pregunta que sabe
responder, que no es la que uno tenía en la cabeza.

Qué hacer si aparece una curva:

1. **No reportar "no hay relación".** Hay relación, y es fuertísima. Lo que no hay es relación lineal.
2. Describir la forma en palabras y mostrar el gráfico. Un scatter con una curva evidente es más
   informativo que cualquier coeficiente.
3. Si la relación es monótona pero no recta (siempre sube, aunque sea curva), existe la **correlación
   de Spearman**: es la misma idea aplicada sobre los rangos en vez de sobre los valores. Se pide con
   `serie_a.corr(serie_b, method='spearman')`.

Y la moraleja general, que es la del paso 2 del marco: **el número nunca reemplaza al gráfico.**

</details>

### 3.5 Lo que r no ve (2): un solo outlier

Segundo error provocado. La celda de abajo fabrica 100 pares de números **al azar**, sin ninguna
relación entre ellos. Después agrega **un** punto lejano, y vuelve a calcular.

`np.random.default_rng(42)` crea un generador de números aleatorios con semilla fija, para que a todos
les salga lo mismo.

In [ ]:
generador = np.random.default_rng(42)
a = generador.normal(0, 1, 100)
b = generador.normal(0, 1, 100)

# El mismo par de nubes, con un solo punto extremo agregado al final
a_con_outlier = np.append(a, 20)
b_con_outlier = np.append(b, 20)

r_sin = np.corrcoef(a, b)[0, 1]
r_con = np.corrcoef(a_con_outlier, b_con_outlier)[0, 1]

fig, ejes = plt.subplots(1, 2, figsize=(13, 5))
ejes[0].scatter(a, b, s=20, color='steelblue')
ejes[0].set_title(f'100 puntos al azar: r = {r_sin:.3f}')
ejes[0].set_xlabel('a')
ejes[0].set_ylabel('b')

ejes[1].scatter(a_con_outlier, b_con_outlier, s=20, color='indianred')
ejes[1].set_title(f'Los mismos, más UN punto: r = {r_con:.3f}')
ejes[1].set_xlabel('a')
ejes[1].set_ylabel('b')

plt.tight_layout()
plt.show()

print(f'Sin el outlier: r = {r_sin:.3f}  (no hay relación, y es cierto)')
print(f'Con el outlier: r = {r_con:.3f}  (correlación fuerte, y es mentira)')
print('Un punto sobre 101 cambió la conclusión. Ningún error, ninguna advertencia.')

### 3.6 Lo que r no ve (3): los grupos

El tercero no se puede fabricar en dos líneas porque ya está en nuestros datos, y es el hallazgo del
día. Lo dejamos anotado aquí y lo medimos en la sección 7: **r calculado sobre todo el conjunto puede
ser muy distinto del r calculado dentro de cada grupo.** Es el paso 5 del marco.

Las tres juntas dan la regla operativa de la clase:

> Se grafica antes de calcular, y después se parte por grupos. r no ve curvas, no ve outliers y no ve
> grupos.

### 3.7 Lo que r no ve (4): que usted probó mil pares

**El concepto: las correlaciones espurias.** Una correlación espuria es un r alto entre dos variables
que **no tienen ninguna relación**, ni directa ni a través de una tercera. No es un error de cálculo:
es lo que pasa cuando se prueban muchas parejas y se reporta la mejor.

Los ejemplos famosos vienen de tylervigen.com: películas de Nicolas Cage estrenadas por año contra
ahogamientos en piscinas en Estados Unidos, r cercano a **0,67**; consumo de queso per cápita contra
personas que murieron enredadas en sus sábanas, r cercano a **0,95**; doctorados en ingeniería civil
contra consumo de mozzarella, por encima de **0,9**. No hay mecanismo, no hay confusora, no hay nada.
Hay muchas series de datos y alguien que las cruzó todas.

**El mecanismo, medido.** La celda de abajo fabrica 60 columnas de números **puramente al azar**, sin
ninguna relación entre ellas, con la misma cantidad de filas que tienen las series anuales de esos
ejemplos. Después calcula los r de todos los pares —son 1.770— y muestra el más alto.

Ese número es el que un analista descuidado publicaría como hallazgo.

In [ ]:
generador_espurio = np.random.default_rng(7)

# 60 columnas de puro azar, 25 filas cada una: nada tiene que ver con nada
ruido = pd.DataFrame(generador_espurio.normal(0, 1, size=(25, 60)),
                     columns=[f'var_{i:02d}' for i in range(60)])

matriz_ruido = ruido.corr()
pares_ruido = []
for i in range(len(matriz_ruido.columns)):
    for j in range(i + 1, len(matriz_ruido.columns)):
        pares_ruido.append(abs(matriz_ruido.iloc[i, j]))

pares_ruido = pd.Series(pares_ruido)

print(f'Pares evaluados: {len(pares_ruido):,}')
print(f'La correlación más alta encontrada: r = {pares_ruido.max():.3f}')
print(f'Pares con |r| por encima de 0,5:    {(pares_ruido > 0.5).sum()}')
print(f'Pares con |r| por encima de 0,4:    {(pares_ruido > 0.4).sum()}')
print()
print('Todo esto salió de números aleatorios. Ninguna de esas relaciones existe.')

**La lección operativa, y es de método, no de código.** Con 10 variables hay 45 pares; con 48
columnas, como el dataset del reto de hoy, hay más de mil. Encontrar un r de 0,9 entre dos de mil pares
no es un hallazgo: es aritmética.

> **Primero la hipótesis, después el número.** Si el número llegó primero, hay que poder contar el
> mecanismo antes de publicarlo.

Y de ahí sale la pregunta que hay que hacerse siempre que uno vea un r alto en un heatmap grande:
*¿yo estaba buscando este par, o me lo encontré rastrillando?*

### 3.8 El primer coeficiente sobre el dataset real: el año contra el consumo

Bajemos de los ejemplos fabricados al archivo de Empocaldas. La pregunta más obvia que alguien haría
con nueve años de facturación en la mano es si el consumo subió con el tiempo, y el instrumento que
acabamos de aprender responde eso con una sola línea: `serie_a.corr(serie_b)`.

**Antes de mirar la salida, prediga.** ¿Espera una correlación fuerte, moderada o nula entre el año y
el consumo de agua en Caldas entre 2015 y 2023? Predecir y fallar enseña más que acertar por
casualidad, y aquí el resultado no es el que casi nadie espera.

In [ ]:
r_anio_consumo = df['ANIO'].corr(df['CONSUMO_ACUEDUCTO'])

print(f'r(ANIO, CONSUMO_ACUEDUCTO) = {r_anio_consumo:.3f}')
print()
print('Consumo total de acueducto facturado por año:')
print(df.groupby('ANIO')['CONSUMO_ACUEDUCTO'].sum().round(0).to_string())

**Pregunta de interpretación 3.** El r entre el año y el consumo es 0,023, prácticamente cero.
¿Se puede concluir que el consumo de agua en Caldas no cambió entre 2015 y 2023? Mire la tabla de
totales por año antes de responder.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

No. Un r cercano a cero dice que **no hay tendencia lineal creciente ni decreciente** en los datos
tal como están puestos, y nada más. La tabla de totales por año lo confirma de otra manera: los años
suben, bajan y vuelven a subir, sin una pendiente sostenida. Un r de 0,023 es la manera de r de decir
"la mejor recta que puedo trazar es horizontal", no "aquí no pasó nada".

Hay una segunda razón, más de fondo, y es la que importa para el proyecto de su equipo: **la fila de
este dataset no es un año, es un municipio-estrato-mes.** Al correlacionar `ANIO` contra
`CONSUMO_ACUEDUCTO` fila por fila, se está pidiendo que el año explique la variación entre municipios
y entre estratos, que es donde vive casi toda la varianza del archivo. El ruido entre grupos entierra
cualquier señal temporal que pudiera existir.

Lo correcto cuando una de las dos variables es el tiempo es **agregar primero y graficar después**:
`df.groupby('ANIO')['CONSUMO_ACUEDUCTO'].sum()` y mirar la serie. La correlación de Pearson no es la
herramienta para tendencias temporales; es la herramienta para relaciones entre dos magnitudes
medidas sobre la misma unidad de observación.

</details>

**Pregunta de interpretación 4.** Suponga que ese r hubiera dado 0,60 en vez de 0,023. ¿Podría
escribir en el informe que "el paso del tiempo hace subir el consumo de agua"? ¿Qué otra cosa estaría
midiendo ese número?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

No podría, y por dos razones distintas. La primera es la regla general de la clase: correlación no es
causalidad, y aquí el "mecanismo" sería absurdo, porque el calendario no consume agua. El tiempo casi
nunca es una causa; es el eje sobre el que se mueven las causas de verdad.

La segunda es concreta y es la que habría que investigar: un r de 0,60 entre año y consumo estaría
midiendo, con toda probabilidad, el **crecimiento del número de suscriptores facturados**. Empocaldas
conecta usuarios nuevos cada año, y más usuarios facturados significa más metros cúbicos facturados
aunque cada hogar gaste exactamente lo mismo. El número de suscriptores sería la variable confusora
clásica: está detrás del año y del consumo a la vez.

La contramedida está a una línea de distancia y es la misma de toda la clase: mirar
`PROMEDIO_ACUEDUCTO`, que es consumo por suscriptor y no crece solo porque crezca la base de
clientes. Si el consumo **por suscriptor** también subiera con los años, ahí sí habría algo que
explicar.

</details>

---

## 4. Matriz de correlación y heatmap

Hasta aquí, un par a la vez. `df.corr(numeric_only=True)` calcula **todos los r contra todos**, de una.

Dos cosas de la matriz:

- La **diagonal siempre vale 1**: toda variable correlaciona perfecto consigo misma. No es un
  hallazgo, es aritmética.
- Es **simétrica**: r(a, b) = r(b, a). La mitad de arriba y la de abajo dicen lo mismo.

**Y el parámetro `numeric_only=True`, que no es decoración.** La correlación necesita números; `MES` y
`MUNICIPIO` son texto. Sin ese parámetro, pandas intenta convertir el texto a número y revienta. La
celda de abajo lo provoca a propósito, con `try / except`, para que vea el mensaje ahora y no en el
reto.

---

### Para entender qué está pasando · Qué hace `.corr()` por dentro

Tres cosas que hay que saber de esta función, porque las tres deciden si el resultado sirve.

**1. Qué devuelve.** Un DataFrame cuadrado, con las mismas etiquetas en filas y en columnas, donde la
celda `(fila, columna)` es el r de esas dos variables. Es una tabla como cualquier otra: se puede
indexar con `.loc`, ordenar y filtrar. No es un objeto especial.

**2. Sobre qué columnas opera, y por qué ignora las categóricas.** Solo sobre las numéricas. Y no es
una limitación de pandas: **la correlación de Pearson no está definida para una variable categórica.**
El cálculo de la sección anterior necesita restar la media y dividir por la desviación estándar, y
`MUNICIPIO` no tiene promedio. Preguntar "¿cuánto correlaciona el municipio con el consumo?" no es una
pregunta difícil: es una pregunta mal planteada. Cuando una de las dos variables es categórica, la
herramienta es otra, y es la sección 5.

Cuidado con el reverso, que es la trampa de la sección 4.5: que una columna sea numérica **no**
significa que sea una cantidad. `.corr()` no distingue una cantidad de un código.

**3. Qué hace con los nulos.** Los descarta **por pares** (*pairwise*): para calcular el r entre dos
columnas usa solo las filas donde **las dos** tienen dato, e ignora el resto. No avisa. La consecuencia
incómoda es que **cada celda de la matriz puede estar calculada sobre un número de filas distinto**, y
la matriz no lo muestra por ningún lado. Con muchos nulos, dos celdas vecinas pueden no ser
comparables entre sí.

Nuestro `df` de hoy no tiene nulos porque los quitamos en la carga. La celda de abajo lo demuestra con
un ejemplo diminuto, que es la única forma de verlo.

In [ ]:
ejemplo_1 = pd.Series([1.0, 2.0, 3.0, None, 5.0])
ejemplo_2 = pd.Series([2.0, 4.0, None, 8.0, 10.0])

filas_usables = (ejemplo_1.notna() & ejemplo_2.notna()).sum()

print('Serie 1:', ejemplo_1.tolist())
print('Serie 2:', ejemplo_2.tolist())
print()
print('Filas en total:                    ', len(ejemplo_1))
print('Filas donde LAS DOS tienen dato:   ', filas_usables)
print('r que devuelve .corr():            ', round(ejemplo_1.corr(ejemplo_2), 3))
print()
print('El r se calculó sobre', filas_usables, 'filas, no sobre', len(ejemplo_1), 'y nada lo dijo.')

In [ ]:
try:
    df.corr()
except Exception as error:
    print('Sin numeric_only ->', type(error).__name__)
    print('  ', error)

print()
print("El mensaje nombra el valor que no pudo convertir. Es la pista: hay texto en la tabla.")

In [ ]:
matriz = df.corr(numeric_only=True)

matriz.round(2)

### 4.2 El heatmap

La matriz con números es correcta pero cuesta leerla: son 49 celdas y todas se parecen. El **mapa de
calor** (*heatmap*) la pinta, y la vuelve legible por bloques.

Los parámetros de `sns.heatmap()` que importan:

- `annot=True` — escribe el número dentro de cada celda. Sin esto es decoración.
- `cmap='coolwarm'` — la paleta: azul para negativo, rojo para positivo.
- `center=0` — **el más importante.** Pone el color neutro en el cero.
- `vmin=-1, vmax=1` — fija la escala al rango real de r, para que dos heatmaps distintos se puedan
  comparar entre sí.
- `fmt='.2f'` — dos decimales.
- `square=True` — celdas cuadradas, que se leen mejor.

In [ ]:
plt.figure(figsize=(11, 9))
sns.heatmap(matriz,
            annot=True,
            cmap='coolwarm',
            center=0,
            vmin=-1, vmax=1,
            fmt='.2f',
            square=True,
            linewidths=0.5)
plt.title('Mapa de calor de correlaciones — consumo de agua Empocaldas', fontsize=13)
plt.tight_layout()
plt.show()

### 4.3 El error que no avisa: un heatmap sin `center=0`

Tercer error provocado, y este es de los peligrosos porque el gráfico sale bonito.

Sin `center=0` ni `vmin`/`vmax`, seaborn reparte toda la escala de color entre el mínimo y el máximo
**de esa matriz**. Si la matriz no tiene correlaciones negativas, el color neutro deja de estar en el
cero y se corre hacia arriba: un r de 0,3 puede verse tan intenso como uno de 0,9, y la persona que
mira el gráfico —que es la que decide— lee una relación fuerte donde hay una moderada.

Los dos heatmaps de abajo tienen **exactamente los mismos números**. Compárelos.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(matriz, annot=True, cmap='coolwarm', fmt='.2f',
            square=True, linewidths=0.5, ax=ejes[0], cbar=True)
ejes[0].set_title('SIN center=0: la escala engaña')

sns.heatmap(matriz, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            fmt='.2f', square=True, linewidths=0.5, ax=ejes[1], cbar=True)
ejes[1].set_title('CON center=0, vmin=-1, vmax=1: la escala dice la verdad')

plt.tight_layout()
plt.show()

### 4.4 Cómo se lee un heatmap

Se lee por bloques de color, no celda por celda.

1. **El bloque rojo intenso arriba a la izquierda**: suscriptores y consumos totales, todos entre 0,73
   y 0,88. Todas son medidas de "qué tan grande es este grupo". Correlacionan entre sí porque miden
   casi lo mismo.

2. **`SUSCRIPTORES_ACUEDUCTO` contra `SUSCRIPTORES_ALCANTARILLADO` da 0,88**, la más alta del dataset.
   **Cuidado con celebrarla.** Casi todo suscriptor de acueducto es también suscriptor de
   alcantarillado: son dos formas de contar la misma gente. Un r alto entre dos maneras de medir lo
   mismo es una **tautología**, no un hallazgo.

3. **La fila y la columna de `ANIO` son pálidas**: cerca de cero contra todo. En nueve años el consumo
   total no tiene tendencia lineal.

4. **Las columnas `PROMEDIO` son pálidas frente a suscriptores** y moderadas entre sí (0,58). Otra
   vez: los cocientes viven en otro mundo que las sumas.

**Pregunta de interpretación 5.** De los pares con r alto en este heatmap, ¿cuáles son hallazgos
reales y cuáles son tautologías? Nombre por lo menos uno de cada tipo, y explique con qué criterio
los separó.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Tautología:** `SUSCRIPTORES_ACUEDUCTO` contra `SUSCRIPTORES_ALCANTARILLADO`, con 0,88, el r más
alto de la matriz. En un municipio de Caldas, casi todo el que tiene acueducto tiene alcantarillado:
las dos columnas cuentan a la misma gente con dos nombres distintos. Saberlo no cambia ninguna
decisión de la empresa, y reportarlo como "el hallazgo más fuerte del análisis" es el error más
barato que se puede cometer con una matriz.

**Hallazgo real, aunque modesto:** `SUSCRIPTORES_ACUEDUCTO` contra `CONSUMO_ACUEDUCTO`, con 0,749.
Aquí las dos columnas miden cosas distintas —cuánta gente hay y cuánta agua se factura— y el número
sí habilita una decisión: se puede estimar consumo a partir de la base de suscriptores para
proyectar demanda. En la sección 7 vamos a ver que esa estimación funciona bien en unos grupos y se
cae en otros, que es exactamente el tipo de matiz que un hallazgo real permite investigar y una
tautología no.

**El criterio para separarlos** no es el valor de r: es preguntarse si las dos columnas son
mediciones independientes de fenómenos distintos, o dos formas de anotar lo mismo. Y la segunda
pregunta, la que decide si vale la pena escribirlo: **¿qué decisión cambia si lo sé?** Si la
respuesta es "ninguna", el número es correcto y el hallazgo es nulo.

</details>

### 4.4.1 Leer un número puntual de la matriz

Una matriz de correlación es un DataFrame como cualquier otro: sus filas y sus columnas tienen
nombre. `.loc['etiqueta_fila', 'etiqueta_columna']` saca una celda por sus nombres, sin recalcular
nada. Vale la pena verlo una vez: cuando en el reto tenga que reportar "las tres correlaciones más
fuertes", va a estar leyendo de una matriz, no calculando pares sueltos.

In [ ]:
r_suscriptores = matriz.loc['SUSCRIPTORES_ACUEDUCTO', 'SUSCRIPTORES_ALCANTARILLADO']

print(f'r(suscriptores acueducto, suscriptores alcantarillado) = {r_suscriptores:.3f}')
print()
print('Suscriptores de acueducto y de alcantarillado, primeras filas:')
print(df[['MUNICIPIO', 'ESTRATO', 'SUSCRIPTORES_ACUEDUCTO',
          'SUSCRIPTORES_ALCANTARILLADO']].head(8).to_string(index=False))

**Pregunta de interpretación 6.** Un compañero propone quitar del análisis una de las dos columnas
de suscriptores, porque "correlacionan 0,88 y sobra una". ¿Está de acuerdo? ¿Qué se gana y qué se
pierde al eliminarla?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Como decisión analítica tiene bastante sentido, y hasta tiene nombre: dos columnas que miden casi lo
mismo son **información redundante**, y cargarlas las dos infla artificialmente la impresión de que
el dataset tiene muchas variables cuando en realidad tiene menos. En la clase 14, cuando entremos en
modelos, esa redundancia además molesta de verdad.

Lo que se pierde es justamente el 12% en el que las dos columnas **no** coinciden, y ese resto puede
ser lo interesante: los sitios donde hay acueducto pero no alcantarillado son un dato de cobertura de
servicios públicos, no ruido. Si la pregunta del análisis es sobre consumo, sobra una; si es sobre
cobertura, la diferencia entre las dos columnas es el hallazgo.

La regla operativa: **un r alto no autoriza a borrar una columna, autoriza a preguntarse por qué hay
dos.** Y la decisión se toma mirando la pregunta que se quiere responder, no el número. Nótese que
esto es lo contrario de lo que haría el reflejo automático de "quitar lo redundante": el analista
decide, el coeficiente solo avisa.

</details>

### 4.5 El error más sutil de la clase: correlacionar lo que no es una cantidad

Cuarto error provocado, y el que más se cuela en los entregables.

`df.corr(numeric_only=True)` toma **todas** las columnas numéricas. Pero *numérico* no es lo mismo que
*cantidad*. Un código de departamento, un número de identificación, una latitud, el año como etiqueta:
todos son números y ninguno es una magnitud que tenga sentido promediar o correlacionar.

La celda de abajo agrega una columna con el número de fila —el ejemplo más puro de identificador— y la
mete en la matriz. El resultado es un número perfectamente calculado que no significa nada.

En el reto de hoy esto importa de verdad: el dataset del Saber Pro tiene `cod_dep_nac`,
`snies_progra`, `lat_ciu_nac` y varios más. Son numéricos y no son cantidades.

In [ ]:
ensayo_ids = df.copy()
ensayo_ids['ID_REGISTRO'] = range(len(ensayo_ids))

correlaciones_id = ensayo_ids.corr(numeric_only=True)['ID_REGISTRO'].round(3)

print('Correlaciones del número de fila contra todo lo demás:')
print(correlaciones_id.to_string())
print()
print('Ninguno de estos números significa nada: ID_REGISTRO es una etiqueta, no una cantidad.')
print('Y sin embargo pandas los calculó sin una queja.')

---

## 5. Categórica contra numérica

Cambio de caso en la tabla del paso 1. Cuando una de las dos variables es **categórica**, el scatter
no sirve: no hay eje numérico donde poner las categorías. La tabla dice **boxplot** o **barras de
medias**.

**El boxplot ya lo conoce** de la clase 4: caja, mediana, cuartiles, bigotes y outliers. Lo nuevo hoy
no es el gráfico, es **cuándo se usa**: es la respuesta de la tabla del paso 1 cuando una de las dos
variables es categórica. Si necesita repasar cómo se lee una caja, está en el demo de la clase 4,
sección 5.

**Qué son las barras de medias.** Una barra por categoría, con la altura de su media. Más fácil de
leer para cualquier audiencia, y esconde toda la dispersión.

Casi siempre conviene el boxplot. Las barras se usan cuando la audiencia no lee boxplots, y con una
nota advirtiendo lo que ocultan.

**Nota sobre un aviso que puede aparecer.** Al dibujar boxplots, seaborn puede imprimir un
`MatplotlibDeprecationWarning` sobre el parámetro `vert`. Viene de dentro de seaborn, no del código de
este cuaderno, y no afecta ningún resultado. Ignórelo.

In [ ]:
orden_estratos = ['Estrato1', 'Estrato2', 'Estrato3', 'Estrato4', 'Estrato5',
                  'Estrato6', 'Comercial', 'Industrial', 'Publico / Oficial']

plt.figure(figsize=(13, 6))
sns.boxplot(x='ESTRATO', y='PROMEDIO_ACUEDUCTO', data=df, order=orden_estratos)
plt.xlabel('Estrato o categoría')
plt.ylabel('Consumo por suscriptor (m3)')
plt.title('Consumo por suscriptor según estrato')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

El boxplot de Industrial se sale del gráfico: su escala es tan distinta que aplasta a todos los demás
contra el piso. **Eso es el hallazgo**, pero conviene mirar también sin Industrial para poder comparar
los residenciales entre sí.

### 5.2 El error que no avisa, versión pandas 3: modificar una copia

Quinto error provocado, y aquí hay que ser preciso porque cambió con la versión.

Para quedarse con los residenciales uno escribe `residenciales = df[...]`, que devuelve una **tabla
nueva e independiente**. Si después le asigna un valor a una columna de esa tabla nueva, el `df`
original **no se entera**.

Lo que cambió: las versiones viejas de pandas avisaban con un `SettingWithCopyWarning`. **En pandas 3
ese aviso ya no existe.** La asignación no falla, no advierte y no se propaga: el síntoma es el
silencio absoluto.

La regla operativa es corta:

- Si quiere una tabla aparte para analizar, filtre y **use `.copy()`** para dejar claro que es suya.
- Si lo que quiere es modificar el original, la herramienta es `.loc`:
  `df.loc[condicion, 'columna'] = valor`.

La celda de abajo trabaja sobre un `ensayo` para no dañar nuestro `df`. Mire los números, no el código.

In [ ]:
ensayo = df.copy()

# Intento 1: modificar el resultado de un filtro. No toca el original, y no avisa.
solo_industrial = ensayo[ensayo['ESTRATO'] == 'Industrial']
solo_industrial['PROMEDIO_ACUEDUCTO'] = 0

print('Máximo de Industrial en el original después del "cambio":',
      ensayo[ensayo['ESTRATO'] == 'Industrial']['PROMEDIO_ACUEDUCTO'].max())
print('Ni un error, ni una advertencia, ni un cambio.')

# Intento 2: modificar el original de verdad, con .loc
ensayo.loc[ensayo['ESTRATO'] == 'Industrial', 'PROMEDIO_ACUEDUCTO'] = 0
print()
print('Con .loc, el máximo de Industrial en el original es:',
      ensayo[ensayo['ESTRATO'] == 'Industrial']['PROMEDIO_ACUEDUCTO'].max())
print('Y nuestro df sigue intacto:',
      df[df['ESTRATO'] == 'Industrial']['PROMEDIO_ACUEDUCTO'].max())

In [ ]:
residenciales = df[~df['ESTRATO'].isin(['Industrial', 'Publico / Oficial'])].copy()

plt.figure(figsize=(12, 6))
sns.boxplot(x='ESTRATO', y='PROMEDIO_ACUEDUCTO', data=residenciales,
            order=['Estrato1', 'Estrato2', 'Estrato3', 'Estrato4',
                   'Estrato5', 'Estrato6', 'Comercial'])
plt.xlabel('Estrato o categoría')
plt.ylabel('Consumo por suscriptor (m3)')
plt.title('Consumo por suscriptor, sin Industrial ni Público (para poder comparar)')
plt.tight_layout()
plt.show()

### 5.3 El mismo dato, en barras de medias

`sns.barplot()` dibuja la media de cada categoría. La barrita vertical encima de cada barra es el
intervalo de confianza de esa media: qué tan segura está la estimación del promedio.

In [ ]:
plt.figure(figsize=(13, 6))
sns.barplot(x='ESTRATO', y='PROMEDIO_ACUEDUCTO', data=df,
            order=orden_estratos, errorbar=('ci', 95))
plt.xlabel('Estrato o categoría')
plt.ylabel('Consumo medio por suscriptor (m3)')
plt.title('Consumo medio por suscriptor según estrato')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

**Pregunta de interpretación 7.** Compare el boxplot completo con las barras de medias. ¿Qué
información se pierde al pasar de uno al otro? ¿En qué situación usaría cada uno?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Se pierde toda la **dispersión**. Las barras dicen dónde está el centro de cada grupo y nada más: no
se ve la mediana, no se ven los cuartiles, no se ven los outliers, y no se ve que dentro de Industrial
haya una variabilidad enorme mientras que dentro de Estrato6 los valores están muy juntos. Dos grupos
con la misma media y dispersiones completamente distintas producen barras idénticas.

Se gana legibilidad. Una barra la lee cualquiera; un boxplot hay que saber leerlo, y la mitad de una
audiencia no técnica confunde los bigotes con barras de error.

El criterio práctico:

- **Boxplot** cuando importa la dispersión, o sea casi siempre en análisis. Es lo que uno usa para
  entender.
- **Barras de medias** cuando lo único que importa es comparar niveles y la audiencia no lee
  boxplots, o sea a veces en comunicación. Y con una nota que diga que esconde la dispersión.

Una observación honesta sobre este gráfico en particular: las barras aquí son casi inservibles, porque
Industrial tiene una media tan alta que aplasta a todas las demás a una franja de un milímetro. Ese es
otro criterio: cuando los órdenes de magnitud son muy distintos, ni barras ni boxplot conjunto
funcionan, y hay que separar.

</details>

### 5.4 Otra categórica: el mes del año

El estrato no es la única categórica del archivo. `MES` también lo es, y la pregunta que habilita es
distinta: ¿el consumo por suscriptor cambia según la época del año? En un país con estaciones la
respuesta sería obvia; en Caldas no lo es.

Mismo par de tipos, así que mismo gráfico. Dos detalles del código de abajo, porque son los que va a
tener que copiar en el reto: los nombres de los meses son largos y se encimarían, de ahí
`plt.xticks(rotation=45)`; y el gráfico lleva **título y las dos etiquetas de eje**, que no es un
adorno sino lo que piden las rúbricas de los momentos evaluativos. Un gráfico sin ejes etiquetados
no se puede leer sin el código al lado, y quien lo evalúa no tiene el código al lado.

**Antes de mirar la salida, prediga:** ¿espera que el consumo por suscriptor cambie según el mes?

In [ ]:
plt.figure(figsize=(13, 6))
sns.boxplot(x='MES', y='PROMEDIO_ACUEDUCTO', data=df)
plt.xlabel('Mes')
plt.ylabel('Consumo por suscriptor (m3)')
plt.title('Consumo por suscriptor según el mes del año')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Media y mediana del consumo por suscriptor, por mes:')
print(df.groupby('MES')['PROMEDIO_ACUEDUCTO'].agg(['mean', 'median', 'count']).round(2).to_string())

**Pregunta de interpretación 8.** Las cajas de los doce meses quedaron aplastadas contra el eje
horizontal, con una nube de puntos disparada hacia arriba, y las medianas van de 8,4 a 10,0 m3.
¿Se puede concluir de aquí que no hay estacionalidad en el consumo de agua en Caldas?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Se puede concluir algo más modesto y más honesto: **en este archivo, tal como está agregado, el mes
no separa los grupos.** Las medianas se mueven en un rango de 1,6 m3 sobre valores de 8 a 10, o sea
menos de lo que varía un mismo municipio de un año a otro. Enero es el mes más alto y junio el más
bajo, pero la diferencia es pequeña frente a la dispersión que hay dentro de cada mes.

Antes de firmar "no hay estacionalidad" hay que notar por qué el gráfico se ve así: las cajas están
aplastadas porque los outliers industriales (cientos de m3 por suscriptor) estiran el eje vertical.
El boxplot está mostrando la escala del eje más que la diferencia entre meses. La contramedida es la
de siempre en esta clase: **partir por grupo antes de concluir**, o sea repetir el gráfico solo con
estratos residenciales, donde los órdenes de magnitud son comparables.

Y hay un límite de los datos que ninguna técnica arregla: aquí una fila es un municipio-estrato-mes
facturado, y la facturación no coincide exactamente con el consumo del mes calendario. Si de verdad
importara la estacionalidad, ese es el primer supuesto que habría que verificar con la empresa.

</details>

**Pregunta de interpretación 9.** En la tabla impresa, mayo tiene la **media** más alta de todo el
año (130,4 m3) y a la vez la **mediana** más baja (8,4 m3). ¿Cómo pueden ser ciertas las dos cosas, y
cuál de los dos números reportaría en un informe sobre el consumo típico?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Son ciertas las dos porque miden cosas distintas, y este es el ejemplo de libro de la clase 4: la
media se deja arrastrar por los valores extremos y la mediana no. Basta con que unos pocos grupos
industriales o comerciales hayan facturado consumos enormes en mayo para que la media del mes se
dispare, mientras la mitad de las filas siguen estando por debajo de 8,4 m3. La media dice dónde
está el centro de masa; la mediana, dónde está la fila del medio.

Para "el consumo típico" se reporta la **mediana**, y se dice que es la mediana. Reportar 130 m3 como
consumo típico de mayo sería falso para el 99% de los suscriptores de Caldas. La media no se tira a
la basura, eso sí: sirve para el total facturado, que es lo que le importa a la tesorería de la
empresa, porque la media multiplicada por el número de grupos reconstruye la suma y la mediana no.

La lección que se lleva al proyecto: **cuando media y mediana se separan tanto, el hallazgo es esa
separación.** Es el aviso de que hay un subgrupo con un comportamiento completamente distinto metido
en la misma tabla, y en este dataset ya sabemos cuál es: el sector industrial. Buscarlo es más útil
que elegir entre los dos números.

</details>

### 5.5 La comparación en números: media y conteo por grupo

Los gráficos muestran; los números fijan. Antes de pasar al chequeo de Simpson hay que poder poner el
nivel de cada grupo en una tabla.

`.agg(['mean', 'count'])` pide dos resúmenes de un golpe sobre el mismo GroupBy. Y el conteo no es un
adorno: **una media sin su conteo no se puede interpretar.** Una media calculada sobre 30 filas y una
calculada sobre 2.400 se ven idénticas en pantalla y no valen lo mismo.

In [ ]:
medias_estrato = df.groupby('ESTRATO')['PROMEDIO_ACUEDUCTO'].agg(['mean', 'count'])

print(medias_estrato.round(2).to_string())

**Pregunta de interpretación 10.** Industrial promedia 586,8 m3 por suscriptor y Estrato6 promedia
6,1: casi cien veces más. Los dos grupos tienen un conteo parecido (2.286 y 2.324 filas). ¿Qué
consecuencia tiene esa diferencia de escala para cualquier número que se calcule sobre el dataset
completo?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

La consecuencia es que **todo estadístico global del archivo va a estar contando, sobre todo, al
sector industrial**, aunque sea una novena parte de las filas. El promedio general de consumo por
suscriptor no describe a ningún suscriptor real: está entre dos mundos que no se tocan. Es el mismo
fenómeno que aplastó el boxplot de la sección anterior y que abrió en abanico el scatter de la
sección 2.

Para la correlación, que es el tema de hoy, la consecuencia es más específica y es la que prepara la
sección 7: cuando hay grupos con escalas tan distintas, el r global termina midiendo **la diferencia
de nivel entre los grupos** en vez de la relación dentro de cada uno. Ese es el mecanismo exacto de
la paradoja de Simpson, y aquí están sus dos ingredientes servidos: grupos en niveles muy distintos y
tamaños que no se compensan.

Lo que se hace al respecto no es borrar Industrial: es **declarar el grupo y reportar por grupo**. Un
análisis que dice "el consumo por suscriptor va de 6 m3 en Estrato6 a 587 en Industrial, así que
todos los números siguientes se reportan por estrato" es más útil, y más difícil de refutar, que uno
que promedia todo y produce un número que no le sirve a nadie.

</details>

---

## 6. Multivariado: el pair plot

Hasta aquí se miraron pares. El **pair plot** mira todos los pares a la vez: una cuadrícula donde cada
celda es el scatter de dos variables, y la diagonal muestra la distribución de cada variable sola.

Es el gemelo visual de la matriz de correlación. Donde la matriz da un número, el pair plot da la
**forma**, y la forma es lo que r no ve.

**Siempre sobre una muestra.** Con 21.422 filas y varias variables, seaborn tiene que dibujar cientos
de miles de puntos, se demora una eternidad y produce manchas ilegibles. Con 800 filas se ve mejor y
tarda un segundo.

In [ ]:
variables_numericas = ['SUSCRIPTORES_ACUEDUCTO', 'CONSUMO_ACUEDUCTO', 'PROMEDIO_ACUEDUCTO']

muestra_pair = df.sample(800, random_state=42)

sns.pairplot(muestra_pair, vars=variables_numericas, height=2.6,
             plot_kws={'alpha': 0.4, 's': 14})
plt.suptitle('Pair plot: todos los pares a la vez (muestra de 800)', y=1.02)
plt.show()

### 6.2 El pair plot coloreado por categoría

`hue='ESTRATO'` pinta cada punto según su grupo. Ahí el pair plot deja de ser un resumen y pasa a ser
una herramienta de **descubrimiento**: si los grupos se separan visualmente en algún panel, esa
variable categórica está explicando algo que los pares numéricos por sí solos no muestran.

Se usan solo tres estratos para que se distingan los colores.

In [ ]:
tres_estratos = df[df['ESTRATO'].isin(['Estrato2', 'Comercial', 'Industrial'])]
muestra_estratos = tres_estratos.sample(600, random_state=42)

sns.pairplot(muestra_estratos, vars=variables_numericas, hue='ESTRATO',
             height=2.6, plot_kws={'alpha': 0.5, 's': 18})
plt.suptitle('Pair plot por estrato: los grupos se separan', y=1.02)
plt.show()

Mire el panel de `SUSCRIPTORES_ACUEDUCTO` contra `CONSUMO_ACUEDUCTO`. Los puntos de Industrial no
siguen la misma línea que los de Estrato2: están en otra parte del plano y con otra pendiente.

Eso es "lo que r no ve (3)", el de la sección 3.6, ahora visible. La sección que sigue lo mide.

---

## 7. Paso 5 del marco: el chequeo de Simpson

**Qué es la paradoja de Simpson.** Una tendencia que aparece en el conjunto completo puede
**desaparecer o invertirse** cuando se parte por grupos. El caso canónico son las admisiones de
posgrado de Berkeley en 1973: globalmente admitían al 44% de los hombres y al 35% de las mujeres, lo
que parecía discriminación evidente. Al separar por departamento, la mayoría de los departamentos
admitía a las mujeres en igual o mayor proporción: las mujeres postulaban mayoritariamente a los
departamentos más competidos, con tasas bajísimas para todo el mundo. La confusora era el departamento
al que se postula.

**El chequeo, en cuatro pasos:**

1. Calcule la relación en el conjunto completo.
2. Recalcúlela **dentro de cada grupo**.
3. Compare. Si el signo se invierte o la fuerza cambia mucho, hay una confusora estructural.
4. Reporte lo que encontró, **incluso si es "no hay paradoja"**.

Se hace con un bucle `for` sobre los grupos. Podría hacerse con `groupby().apply()`, pero el bucle se
lee mejor y hoy la claridad importa más que la elegancia. El `if len(grupo) > 30` está para no reportar
correlaciones calculadas sobre un puñado de filas, que no significan nada.

### 7.1 La inversión, vista con datos

Antes de correr el chequeo sobre nuestros datos conviene ver el mecanismo aislado, porque enunciado
suena a truco de magia y no lo es.

Los datos de la celda de abajo son **fabricados a propósito**, y se dice explícitamente porque es
material didáctico, no un hallazgo. Son tres municipios inventados, y dentro de cada uno la relación
entre suscriptores y consumo por suscriptor es **positiva**: a más suscriptores, más consumo por
suscriptor. Los tres grupos, por separado, dicen lo mismo.

Lo único que cambia entre municipios es el **nivel**: el municipio grande tiene, en promedio, un
consumo por suscriptor más bajo. Nada más.

Mire qué dice el r calculado sobre los tres juntos.

In [ ]:
generador_simpson = np.random.default_rng(11)

partes = []
# (municipio, centro de suscriptores, centro de consumo por suscriptor)
for nombre, centro_x, centro_y in [('Municipio A', 200, 30),
                                   ('Municipio B', 600, 20),
                                   ('Municipio C', 1000, 10)]:
    suscriptores = centro_x + generador_simpson.normal(0, 60, 80)
    # Dentro del grupo la pendiente es POSITIVA: +0.05 por suscriptor
    consumo = centro_y + 0.05 * (suscriptores - centro_x) + generador_simpson.normal(0, 1, 80)
    partes.append(pd.DataFrame({'MUNICIPIO': nombre,
                                'SUSCRIPTORES': suscriptores,
                                'CONSUMO_POR_SUSCRIPTOR': consumo}))

ficticio = pd.concat(partes, ignore_index=True)

r_junto = ficticio['SUSCRIPTORES'].corr(ficticio['CONSUMO_POR_SUSCRIPTOR'])
print(f'r con los tres municipios JUNTOS: {r_junto:6.3f}')
print()
for nombre, grupo in ficticio.groupby('MUNICIPIO'):
    r_grupo = grupo['SUSCRIPTORES'].corr(grupo['CONSUMO_POR_SUSCRIPTOR'])
    print(f'r dentro de {nombre}: {r_grupo:6.3f}')

In [ ]:
plt.figure(figsize=(10, 6))
for nombre, grupo in ficticio.groupby('MUNICIPIO'):
    plt.scatter(grupo['SUSCRIPTORES'], grupo['CONSUMO_POR_SUSCRIPTOR'],
                s=18, alpha=0.7, label=nombre)

# La recta de la nube completa, ignorando los grupos
pendiente, corte = np.polyfit(ficticio['SUSCRIPTORES'],
                              ficticio['CONSUMO_POR_SUSCRIPTOR'], 1)
equis = np.linspace(ficticio['SUSCRIPTORES'].min(), ficticio['SUSCRIPTORES'].max(), 50)
plt.plot(equis, pendiente * equis + corte, color='black', linewidth=2,
         label='Tendencia de los tres juntos')

plt.xlabel('Suscriptores')
plt.ylabel('Consumo por suscriptor (m3)')
plt.title('Paradoja de Simpson: cada grupo sube, el conjunto baja (datos fabricados)')
plt.legend()
plt.tight_layout()
plt.show()

**Ahí está la criatura.** Dentro de cada color, la nube sube. La línea negra, que es la tendencia de
los tres juntos, baja. Y las dos cosas son ciertas al mismo tiempo: no hay ningún error de cálculo.

**El mecanismo, en una frase:** cuando los grupos están en niveles distintos y además se distribuyen
de forma desbalanceada a lo largo del eje horizontal, el r global termina midiendo **la diferencia
entre grupos** en vez de la relación **dentro** de cada grupo. La variable que define el grupo es la
confusora, y si no está en el análisis, el número la absorbe.

**Por qué importa tanto y no es una curiosidad de examen:** la conclusión que uno saca es la contraria.
"Más suscriptores, menos consumo por suscriptor" y "más suscriptores, más consumo por suscriptor" son
recomendaciones de política opuestas, y las dos salen del mismo archivo según se mire.

**Cuál de las dos es la correcta.** Depende de la pregunta, y esa decisión es del analista, no del
código. Si la pregunta es "dentro de un municipio, ¿qué pasa cuando crece el número de suscriptores?",
manda la relación **dentro** de los grupos. Si la pregunta es "¿los municipios grandes consumen más por
suscriptor que los pequeños?", manda la **comparación entre** grupos, que es otra pregunta. Lo que
nunca es correcto es reportar el número global sin haber mirado los grupos.

In [ ]:
x = 'SUSCRIPTORES_ACUEDUCTO'
y = 'CONSUMO_ACUEDUCTO'

r_global = df[x].corr(df[y])
print(f'r GLOBAL entre {x} y {y}: {r_global:.3f}')
print()
print('r DENTRO de cada estrato:')
print('-' * 46)

resultados = []
for estrato in df['ESTRATO'].unique():
    grupo = df[df['ESTRATO'] == estrato]
    if len(grupo) > 30:
        r_grupo = grupo[x].corr(grupo[y])
        resultados.append((estrato, r_grupo, len(grupo)))

for estrato, r_grupo, n in sorted(resultados, key=lambda t: -t[1]):
    marca = '  <-- ojo' if abs(r_grupo - r_global) > 0.3 else ''
    print(f'{estrato:20s} r = {r_grupo:6.3f}   n = {n:5,}{marca}')

### Qué pasó aquí

Dos cosas, y las dos importan.

**Primera: en ocho de los nueve grupos, r es MAYOR que el global.** El global da 0,749 y los grupos dan
entre 0,80 y 0,87. La relación es más limpia **dentro** de cada estrato que en el conjunto. Eso es
firma de variable confusora: el estrato estaba metiendo ruido en la relación global, porque cada
estrato vive en una escala distinta de consumo por suscriptor. Es el abanico que vimos en la sección 2.

**Segunda: Industrial se cae a 0,125.** En el sector industrial, el número de suscriptores no predice
el consumo. Y tiene sentido: un suscriptor industrial puede ser un taller de tres personas o una
embotelladora. Contar cuántos hay no dice nada sobre cuánta agua gastan.

**La conclusión que se lleva:** un modelo que prediga consumo a partir de suscriptores va a funcionar
bien en residencial y mal en industrial. Si uno solo hubiera mirado el 0,749, habría salido convencido
de que funciona en todas partes.

**Un matiz honesto:** técnicamente esto no es la paradoja de Simpson en su versión de manual, porque
ningún signo se invirtió. Los datos reales rara vez dan la inversión espectacular del ejemplo de
Berkeley. Lo que sí dan, y es igual de valioso, es esto: una relación que se comporta distinto en un
grupo que en los demás. Reportar eso es hacer el trabajo completo.

### 7.3 El mismo chequeo sobre el par de r nulo, que sí se invierte

En la sección 3.2 calculamos el r entre `SUSCRIPTORES_ACUEDUCTO` y `PROMEDIO_ACUEDUCTO` y dio
**-0,022**: prácticamente cero, y con signo negativo. La lectura que hicimos entonces fue "el tamaño
del grupo no dice nada sobre cuánto gasta cada suscriptor".

Corramos el paso 5 sobre ese mismo par. Es la parte del cuaderno donde una conclusión que ya habíamos
escrito se cae.

In [ ]:
x2 = 'SUSCRIPTORES_ACUEDUCTO'
y2 = 'PROMEDIO_ACUEDUCTO'

r_global2 = df[x2].corr(df[y2])
print(f'r GLOBAL entre {x2} y {y2}: {r_global2:.3f}')
print()
print('r DENTRO de cada estrato:')
print('-' * 46)

positivos = 0
for estrato in sorted(df['ESTRATO'].unique()):
    grupo = df[df['ESTRATO'] == estrato]
    r_grupo = grupo[x2].corr(grupo[y2])
    if r_grupo > 0:
        positivos += 1
    print(f'{estrato:20s} r = {r_grupo:6.3f}   n = {len(grupo):5,}')

print('-' * 46)
print(f'Grupos con r positivo: {positivos} de {df["ESTRATO"].nunique()}')
print(f'Signo del r global: {"positivo" if r_global2 > 0 else "negativo"}')

**El signo se invierte.** El r global es negativo y **ocho de los nueve estratos dan positivo**. Es la
misma criatura del ejemplo fabricado de la sección 7.1, ahora en datos reales de una empresa de
servicios públicos de Caldas.

El mecanismo es el mismo: cada estrato vive en un nivel de consumo por suscriptor completamente
distinto —Industrial promedia 587 m3 por suscriptor y Estrato6 promedia 6— y a la vez tienen tamaños
de grupo distintos. Al juntarlo todo, el r global termina midiendo esa diferencia de niveles entre
estratos en vez de la relación dentro de cada uno.

**Y ahora el matiz, que es tan importante como el hallazgo.** Los r dentro de los grupos son
**débiles**: 0,129, 0,144, 0,009. El único que se despega es Público / Oficial con 0,396. Así que la
conclusión honesta **no** es "descubrimos que hay una relación positiva fuerte que estaba escondida".
Es esta:

> El r global de -0,022 no significa "no hay relación". Significa que se estaba promediando entre
> grupos que viven en escalas distintas, y ese promedio no responde ninguna pregunta útil. Dentro de
> cada estrato la relación es débil y positiva, salvo en el sector público, donde es moderada.

Escribir la primera versión sería exactamente el pecado que la sección 3.7 enseña a evitar: encontrar
el titular y no la verdad.

**Pregunta de interpretación 11.** ¿Qué diferencia hay entre una **variable confusora** y la
**paradoja de Simpson**? ¿Y si usted corre el chequeo y no encuentra ninguna paradoja, le fue mal?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**No son lo mismo, y confundirlas se nota.** La confusora es la **causa**: una tercera variable que
está detrás de las dos y que explica el patrón. La paradoja de Simpson es el **síntoma**: lo que le
pasa a los números cuando esa confusora está desbalanceada entre los grupos. Simpson es lo que uno
**ve**; la confusora es lo que lo **produce**. Puede haber confusora sin paradoja —es exactamente
nuestro caso de hoy: el estrato es confusora y ningún signo se invirtió— pero no hay paradoja sin
confusora.

**Y no, no le fue mal.** Al contrario: que una relación sobreviva a partirla por grupos la hace **más**
creíble, no menos. El entregable es el **chequeo**, no la paradoja. Un análisis que dice "corrí el
chequeo por cuatro variables distintas y la relación se sostiene en todas" es más fuerte que uno que
no lo corrió.

Forzar los datos hasta que aparezca una paradoja —probar particiones hasta que una dé el titular— es
justamente lo que esta clase enseña a no hacer. Es la misma trampa de las correlaciones espurias, con
otra ropa.

</details>

### 7.4 El mismo chequeo, partido por otra variable

El chequeo de Simpson no se corre una vez: se corre por cada variable que pueda estar detrás de la
relación. El estrato ya lo hicimos; falta la otra gran candidata de este dataset, el **municipio**.

El código es el mismo bucle de la sección 7, cambiando la columna por la que agrupa. En vez de
imprimir dentro del bucle, esta vez los resultados se van guardando en un diccionario
`{municipio: r}` y al final se convierten con `pd.Series(diccionario)`. Guardarlos en una Series es
lo que permite ordenarlos: con 24 municipios la lista es larga, y lo que interesa son los extremos,
que se sacan con `.sort_values()` más `.head()` y `.tail()`.

**Antes de mirar la salida, prediga:** ¿espera que la relación entre suscriptores y consumo cambie
mucho de un municipio de Caldas a otro, o que se mantenga parecida?

In [ ]:
resultados_muni = {}
for municipio in df['MUNICIPIO'].unique():
    grupo = df[df['MUNICIPIO'] == municipio]
    resultados_muni[municipio] = grupo[x].corr(grupo[y])

r_por_municipio = pd.Series(resultados_muni)

print(f'r GLOBAL entre {x} y {y}: {r_global:.3f}')
print(f'Municipios evaluados: {len(r_por_municipio)}')
print()
print('Los 5 municipios con el r más bajo:')
print(r_por_municipio.sort_values().head().round(3).to_string())
print()
print('Los 5 municipios con el r más alto:')
print(r_por_municipio.sort_values().tail().round(3).to_string())

**Pregunta de interpretación 12.** Partido por municipio, el r se mantiene alto y positivo en casi
todos (entre 0,79 y 0,95), salvo Kilómetro 41 con 0,24 y Chinchiná con 0,32. Partido por estrato, en
cambio, Industrial se caía a 0,125. ¿Cuál de las dos variables es la confusora que importa aquí, y en
qué se basa para decidirlo?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

La confusora que importa es el **estrato**, y el criterio no es cuál da los números más llamativos:
es cuál **cambia la conclusión**. Al partir por municipio, la relación se sostiene en 22 de 24 casos
y hasta se refuerza respecto al 0,749 global; la lectura del análisis no cambia. Al partir por
estrato, en cambio, aparece un grupo entero —el industrial— donde la relación simplemente no existe,
y eso sí cambia lo que se puede prometer.

La diferencia de fondo es qué representa cada partición. Los municipios son variantes de la misma
cosa: hogares que consumen agua en cantidades comparables, con más o menos suscriptores. Los estratos
no: mezclan hogares con embotelladoras. La confusora es la variable que **separa poblaciones
distintas**, no la que simplemente parte el archivo en pedazos.

Y los dos municipios raros no se tiran a la basura: son la siguiente pregunta. Kilómetro 41 es un
corregimiento pequeño y Chinchiná tiene actividad cafetera industrial, así que lo más probable es que
en esos dos el mix de estratos sea distinto al del resto, o que haya tan pocas filas que el r sea
inestable. Verificarlo cuesta una línea (`df['MUNICIPIO'].value_counts()`) y es exactamente el tipo
de seguimiento que separa un análisis de un reporte.

</details>

**Pregunta de interpretación 13.** El chequeo por municipio no encontró ninguna paradoja: la
relación se sostuvo. ¿Ese resultado se reporta o se calla? ¿Qué le pasaría a la credibilidad de su
informe si un lector corriera el chequeo por su cuenta y usted no lo hubiera mencionado?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Se reporta, y en la parte visible del informe, no en un anexo. Un chequeo que sale negativo **es** el
resultado: dice que la relación sobrevivió a un intento serio de romperla, y eso la hace más creíble,
no menos. La frase que se escribe es del estilo "la relación entre suscriptores y consumo se sostiene
al partir por municipio (r entre 0,79 y 0,95 en 22 de 24) y se rompe en el sector industrial al
partir por estrato".

Callarlo tiene un costo asimétrico. Si el lector corre el chequeo y le da lo mismo que a usted, no
pasa nada, pero usted perdió la oportunidad de mostrar que hizo el trabajo. Si le da algo distinto
—porque filtró otro periodo, u otro conjunto de municipios— el lector no va a concluir "qué
interesante": va a concluir que usted no lo miró. Un informe que no dice qué chequeos corrió obliga a
confiar en el autor; uno que los dice se puede auditar, y en analítica esa es toda la diferencia.

Hay una trampa simétrica que esta clase también quiere evitar: **buscar particiones hasta que una dé
la paradoja**. Probar seis variables y reportar solo la que invirtió el signo es exactamente el
mecanismo de las correlaciones espurias de la sección 3.7, con otra ropa. Lo que se reporta es el
conjunto de chequeos que se corrió, no el que dio el mejor titular.

</details>

---

## 8. Paso 4 del marco: interrogar

El único paso que no tiene código. Para cada relación que uno encuentre, cuatro preguntas:

1. **¿Tiene sentido lógico?** ¿Puedo contar el mecanismo?
2. **¿El orden temporal funciona?** La causa tiene que ir antes que el efecto.
3. **¿Puede haber una tercera variable escondida?**
4. **¿Podría ser espuria?** ¿Cuántos pares probé antes de encontrar este?

**Por qué importa tanto.** Cuando A y B están correlacionados hay **tres** explicaciones posibles, no
una: A causa B, B causa A, o una tercera variable C causa las dos. El ejemplo del bloque 1 —la venta
de helados y los ahogamientos, con r de 0,85— es la tercera: el calor causa las dos cosas.

Aplicado al hallazgo principal del día:

| Pregunta | Respuesta |
|----------|-----------|
| ¿Tiene sentido? | Sí. Más suscriptores facturados, más metros cúbicos facturados. El mecanismo es contable |
| ¿Orden temporal? | Los suscriptores existen antes de consumir. Va bien |
| ¿Tercera variable? | Sí, y la encontramos: el estrato. Cambia la fuerza de la relación, y en Industrial la destruye |
| ¿Espuria? | No. El mecanismo es directo y no salió de rastrillar mil pares |

Compare con lo que pasaría si hubiéramos reportado el 0,88 entre suscriptores de acueducto y de
alcantarillado: el mecanismo existe, pero es que **son la misma gente contada dos veces**. Correlación
real, hallazgo nulo.

**Pregunta de interpretación 14.** ¿A partir de qué valor una correlación es "significativa"? Y si un
dataset tiene 48 columnas, ¿qué probabilidad hay de encontrar correlaciones altas por puro azar?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**No hay un umbral universal, y quien le dé uno le está mintiendo.** Depende del campo: en física se
exige por encima de 0,99 para tomarse algo en serio; en ciencias sociales un 0,3 ya es relevante; en un
negocio, un 0,2 puede ser accionable si el mecanismo tiene sentido y la decisión que habilita vale la
pena. La regla es no reportar nunca el número solo: reportar qué significa **en ese contexto** y qué
decisión cambia.

**Sobre los mil pares:** con 10 variables hay 45 pares; con 48 columnas, más de mil. Si uno prueba
suficientes pares, **va a encontrar** correlaciones altas por puro azar. Encontrar un r de 0,9 entre
dos de mil pares no es un hallazgo, es aritmética. Ese es exactamente el mecanismo de las
**correlaciones espurias**: películas de Nicolas Cage estrenadas por año contra ahogamientos en
piscinas da r cercano a 0,67, y consumo de queso per cápita contra muertes por enredarse en las
sábanas da 0,95.

La defensa es de método, no de cálculo: **primero la hipótesis, después el número.** Si el número llegó
primero, hay que poder contar el mecanismo antes de publicarlo.

</details>

---

## 9. Punto de control

Este cuaderno no se autocalifica: no hay nada que teclear en él. El punto de control es usted
respondiéndose, sin abrir los desplegables, estas tres preguntas:

1. ¿Sabría elegir el gráfico correcto mirando solo los tipos de las dos variables, y decir qué mide
   el número que va con él?
2. ¿Sabría explicar tres razones distintas por las que un r puede engañar, y qué hace en cada caso?
3. ¿Sabría correr un chequeo de Simpson sobre un par cualquiera y reportar el resultado, salga
   paradoja o no salga?

Si alguna respuesta es "no", no pase de largo: el reto del bloque 3 es donde sí hay que escribir
código, y arranca dando por sabido todo lo de este cuaderno. Levante la mano ahora, que el profesor
está en el salón.

---

## 10. Preguntas que siempre salen

**¿Si dos cosas correlacionan fuerte, no tiene que causar una a la otra?**
No. Hay tres explicaciones: A causa B, B causa A, o una tercera variable C causa las dos. Hay que
descartar las otras dos antes de afirmar causalidad, y en un análisis observacional como este casi
nunca se pueden descartar. La contramedida verbal es una sola pregunta: *¿me cuenta el mecanismo?* Si
no puede contarlo, no lo escribe.

**¿Por qué la diagonal de la matriz siempre da 1?**
Porque toda variable correlaciona perfectamente consigo misma. Si el consumo sube 5, el consumo sube 5.
Reportar la diagonal como "la correlación más fuerte del dataset" es el error más barato que existe.

**¿Cuántas variables meto en un heatmap?**
Máximo unas 10, y elegidas, no todas las numéricas del archivo. Un heatmap con 48 columnas es
decorativo, no informativo: los números no se leen y el color no se distingue. Y ojo con meter
identificadores y coordenadas, que es la sección 4.5.

**¿Un r alto siempre es útil?**
No. `SUSCRIPTORES_ACUEDUCTO` contra `SUSCRIPTORES_ALCANTARILLADO` da 0,88 y no sirve de nada: son casi
la misma medición. Un r alto entre dos formas de medir lo mismo es una tautología. La pregunta útil no
es "¿qué tan alto es?" sino "¿qué decisión cambia si lo sé?".

**¿Y si r es bajo, ya puedo decir que no hay relación?**
No: puede decir que no hay relación **lineal**. Puede haber una relación en U perfectamente clara, como
la de la sección 3.4. Por eso se grafica primero.

**¿Cuál es el marco bivariado de 5 pasos y por qué graficar va antes que calcular?**
Tipar, graficar, cuantificar, interrogar, segmentar. Graficar va antes porque el gráfico detecta las
tres cosas que r no ve: relaciones no lineales, outliers que distorsionan, y agrupamientos que insinúan
una confusora.

**¿Este dataset sirve para el proyecto de mi equipo?**
No. Los CSV de las clases son material de enseñanza, elegidos por lo que permiten enseñar, y varios ni
siquiera llegan a los umbrales del proyecto. El dataset del proyecto lo consigue cada equipo, de la
fuente que quiera, y tiene que poder decir de dónde salió y bajo qué condiciones se puede usar.

---

## Resumen

| Lo que hizo | Con qué |
|-------------|---------|
| Ver una relación entre dos numéricas | `plt.scatter(x, y, alpha=0.15)` |
| Ver la tendencia | `sns.regplot(x=, y=, data=)` |
| Un coeficiente | `serie_a.corr(serie_b)` |
| Todos los coeficientes | `df.corr(numeric_only=True)` |
| Leer una celda de la matriz | `matriz.loc['fila', 'columna']` |
| Pintar la matriz | `sns.heatmap(m, annot=True, center=0, vmin=-1, vmax=1)` |
| Categórica contra numérica | `sns.boxplot(x=, y=, data=, order=)` |
| Lo mismo, para audiencia no técnica | `sns.barplot(x=, y=, data=)` |
| Nivel y tamaño de cada grupo | `df.groupby('col')['num'].agg(['mean', 'count'])` |
| Todos los pares a la vez | `sns.pairplot(df.sample(800), vars=[...], hue=)` |
| Una muestra reproducible | `df.sample(n, random_state=42)` |
| Modificar el original | `df.loc[condicion, 'columna'] = valor` |

**Las siete reglas que no se negocian:**

1. Se grafica antes de calcular. r no ve curvas, no ve outliers y no ve grupos.
2. r va de -1 a +1. El signo es la dirección, el valor absoluto es la fuerza. No es un porcentaje.
3. Correlación no es causalidad. Tres explicaciones posibles, siempre.
4. Un r alto entre dos formas de medir lo mismo es una tautología, no un hallazgo.
5. `center=0` en el heatmap, siempre. Sin eso el color miente.
6. Con una categórica de por medio: boxplot si importa la dispersión, barras si solo importa el nivel.
7. Toda relación se chequea partida por grupo. Que sobreviva la hace más creíble; que se rompa en un
   grupo es un hallazgo en sí mismo.

**Autoevaluación honesta.** Si puede responder que sí a estas cinco, está listo para el reto:

- [ ] Puedo calcular un r entre dos columnas y traducirlo a una frase de fuerza y dirección.
- [ ] Sé decir dos cosas que r **no** ve, y cómo se detectan.
- [ ] Puedo dibujar un heatmap correcto y explicar qué hace `center=0`.
- [ ] Sé elegir el gráfico según los tipos de las dos variables, sin mirar la tabla.
- [ ] Puedo ejecutar un chequeo de Simpson y reportar honestamente el resultado, sea cual sea.

**Siguiente:** bloque 3, el reto sobre los resultados del Saber Pro. Mismo marco, datos que no vio.